In [19]:
# Imports
from lxml import etree
import sumolib
import pandas as pd

In [20]:
NETWORK = "../../sumo/berlin.net.xml"
ADDITIONAL = "termination_points.add.xml"

# Load network
net = sumolib.net.readNet(NETWORK)

# Parse charging stations
tree = etree.parse(ADDITIONAL)
root = tree.getroot()

In [21]:
for i, station in enumerate(root.findall("chargingStation"), start=1):
    station.set("numericId", str(i))

tree.write("termination_points.add.xml",
           encoding="UTF-8",
           xml_declaration=True)

In [25]:
# Implementation doesnt need to run everything twice, could just change file after the fact. 
# # Wanted to automate for replicability.
stations = []

for cs in root.findall("chargingStation"):
    lane = cs.attrib["lane"]
    edge = lane.rsplit("_", 1)[0]   # remove lane index
    stations.append({
        "id": cs.attrib["id"],
        "name": cs.attrib["name"],
        "edge": edge,
        "numericId": cs.attrib["numericId"]
    })

# Create one matrix skipping the first station and one skipping the second
for skip_idx, output_file in [
    (0, "deadhead_time_cicerostrasse.csv"),
    (1, "deadhead_time_muellerstrasse.csv"),
]:
    selected = stations[:skip_idx] + stations[skip_idx + 1:]
    if skip_idx == 0:
        selected[0]["numericId"] = "1"

    matrix = pd.DataFrame(
        index=[s["numericId"] for s in selected],
        columns=[s["numericId"] for s in selected],
        dtype=float,
    )

    rows = []

    for origin in selected:
        fromEdge = net.getEdge(origin["edge"])

        for dest in selected:
            if origin["id"] == dest["id"]:
                continue  # optional: skip trips to itself

            toEdge = net.getEdge(dest["edge"])

            path, cost = net.getFastestPath(fromEdge, toEdge)

            rows.append({
                "FromStopID": origin["numericId"],
                "ToStopID": dest["numericId"],
                "RunTime": round(cost)
            })

    df = pd.DataFrame(rows)
    df.to_csv(output_file, sep=";", index=False)